<a href="https://www.kaggle.com/code/romerolykajoy/ad-campaign-conversion-rates-success-analysis?scriptVersionId=222096699" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
ad = pd.read_csv("/kaggle/input/advertising-campaign-performance-dataset/ad_campaign_performance.csv")

# Data Check

In [ ]:
ad.head()

In [ ]:
ad.isnull().sum()

In [ ]:
ad.duplicated().sum()

In [ ]:
ad.dtypes

In [ ]:
ad.describe()

## Observations

* Click-through Rate and Conversion Rates have **values more than 100%**. Rates should not go above 100%.
* Click-through rate may possibly **not really represent CTR** since CTR's formula should be (Clicks / Impressions) * 100. Since we do not have impressions column, I will just be dropping this column.
* Since the conversion rates go up to max of 1554.12, I will be checking for the rows where there are higher number of conversions versus clicks. (Note: Some possible cause I see is that one user may have several clicks, or this may possibly an entry error. Since this is a synthetic data, it's most likely data generation error)

## Data Correction

In [ ]:
faultyConversionRate = ad[ad['Conversions']>ad['Clicks']]
print(f"There are {faultyConversionRate.shape[0]} rows with faulty conversion / click ratio")

41 rows only take up 4.1% of the data. I will just consider this as data generation error so I will just be dropping these rows.

In [ ]:
#Dropping Faulty Rows

ad = ad.drop(faultyConversionRate.index)

In [ ]:
#Dropping CTR

ad = ad.drop('CTR', axis = 1)

In [ ]:
ad.describe()

# Proportion of the target demographics

In [ ]:
age = ad.groupby('Target_Age').size().reset_index(name='Count')
age['Proportion'] = age['Count'] / ad.shape[0] * 100.00
age

In [ ]:
gender = ad.groupby('Target_Gender').size().reset_index(name='Count')
gender['Proportion'] = gender['Count'] / ad.shape[0] * 100.00
gender

In [ ]:
region = ad.groupby('Region').size().reset_index(name='Count')
region['Propotion'] = region['Count'] / ad.shape[0] * 100.00
region

# Overall Success Rates

## Success Rate Across Platform

In [ ]:
platformSuccess = ad.groupby('Platform')['Success'].mean()*100
platformSuccess = platformSuccess.plot(kind = 'bar')

#Annotate
for bar in platformSuccess.patches:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
    f'{bar.get_height():.2f}%',
    ha = 'center', va = 'bottom',
    fontsize = 7)

#Plot Labels
plt.title('Platform-wise Success Rates')
plt.xlabel('')
plt.ylabel('Success Rate (%)')
plt.xticks(rotation=45)
plt.show()

## Success Rate Across Content-Type

In [ ]:
contentSuccess = ad.groupby('Content_Type')['Success'].mean()*100
contentSuccess = contentSuccess.plot(kind = 'bar')

#Annotate
for bar in  contentSuccess.patches:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
    f'{bar.get_height():.2f}%',
    ha = 'center', va = 'bottom',
    fontsize = 7)

#Plot Labels
plt.title('Content Type Success Rates')
plt.xlabel('')
plt.ylabel('Success Rate (%)')
plt.xticks(rotation=45)
plt.show()

## Success Rates Among Demographics

In [ ]:
platformSuccess = ad.groupby('Target_Gender')['Success'].mean()*100
platformSuccess = platformSuccess.plot(kind = 'bar')

#Annotate
for bar in platformSuccess.patches:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
    f'{bar.get_height():.2f}%',
    ha = 'center', va = 'bottom',
    fontsize = 7)

#Plot Labels
plt.title('Success Rates Across Target Gender')
plt.xlabel('')
plt.ylabel('Success Rate (%)')
plt.xticks(rotation=45)
plt.show()

In [ ]:
platformSuccess = ad.groupby('Target_Age')['Success'].mean()*100
platformSuccess = platformSuccess.plot(kind = 'bar')

#Annotate
for bar in platformSuccess.patches:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
    f'{bar.get_height():.2f}%',
    ha = 'center', va = 'bottom',
    fontsize = 7)

#Plot Labels
plt.title('Success Rates Across Age Group')
plt.xlabel('')
plt.ylabel('Success Rate (%)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
platformSuccess = ad.groupby('Region')['Success'].mean()*100
platformSuccess = platformSuccess.plot(kind = 'bar')

#Annotate
for bar in platformSuccess.patches:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
    f'{bar.get_height():.2f}%',
    ha = 'center', va = 'bottom',
    fontsize = 7)

#Plot Labels
plt.title('Success Rates Across Region')
plt.xlabel('')
plt.ylabel('Success Rate (%)')
plt.xticks(rotation=45)
plt.show()

# Histograms: Conversion Rates, Budget, Duration

## Histogram of Conversion Rates

In [ ]:
plt.hist(ad['Conversion_Rate'], bins = 10, edgecolor = 'black')
plt.title("Conversion Rates")
plt.show()

In [ ]:
from scipy.stats import skew
skew(ad['Conversion_Rate'])

In [ ]:
ad['Conversion_Rate'].mean()

In [ ]:
plt.hist(ad['Budget'], bins = 10, edgecolor = 'black')
plt.title("Budget Histogram")
plt.show()

In [ ]:
plt.hist(ad['Duration'], bins = 10, edgecolor = 'black')
plt.title("Duration Histogram")
plt.show()

## Top 10 Campaigns with Highest Conversion Rates

In [ ]:
ad.nlargest(10, 'Conversion_Rate')

Interesting Note: In previous sections, text-type has the lowest overall success rate among all content types. However,  6 out of the top performing campaigns (in terms of conversion) are text-type contents. It would be ideal for the marketing team to explore what components are in the said campaigns and utilize it in the next text-type campaigns.

## Budget & Conversion Rate
Does increasing the budget lead to a proportional increase in conversions?

In [ ]:
sns.scatterplot(x='Budget',y='Conversion_Rate',data=ad)

plt.title("Scatter Plot of Budget vs Conversion Rate")
plt.xlabel("Budget")
plt.ylabel("Conversion Rate (%)")
plt.show()

In [ ]:
ad['Conversion_Rate'].corr(ad['Budget'])

There is .056 correlation coefficient between budget and conversion rates, indicating almost very weak correlation between the 2 variables. We can also observe a random scatter plot not indicating any direction. 

## Duration & Conversion Rate
Does longer campaigns lead to a proportional increase in conversions?

In [ ]:
sns.scatterplot(x='Duration',y='Conversion_Rate',data=ad)

plt.title("Scatter Plot of Duration vs Conversion Rate")
plt.xlabel("Duration")
plt.ylabel("Conversion Rate (%)")
plt.show()

In [ ]:
ad['Conversion_Rate'].corr(ad['Duration'])

Similarly, duration has very little correlation to conversion rates.

# Conversion Rates & Demographics Data Viz

## Conversion Rates: Content Type and Age Group

In [ ]:
video = ad[ad['Content_Type'] == 'Video'].groupby('Target_Age')['Conversion_Rate'].mean()
image = ad[ad['Content_Type'] == 'Image'].groupby('Target_Age')['Conversion_Rate'].mean()
text = ad[ad['Content_Type'] == 'Text'].groupby('Target_Age')['Conversion_Rate'].mean()
story = ad[ad['Content_Type'] == 'Story'].groupby('Target_Age')['Conversion_Rate'].mean()
carousel = ad[ad['Content_Type'] == 'Carousel'].groupby('Target_Age')['Conversion_Rate'].mean()

age = np.sort(ad['Target_Age'].unique())

x = np.arange(len(age))
w = 0.15

plt.bar(x - 2 * w, video, w, label="Video",edgecolor = "black")
plt.bar(x - w, image, w, label="Image",edgecolor = "black")
plt.bar(x, text, w, label="Text",edgecolor = "black")
plt.bar(x + w, story, w, label="Story",edgecolor = "black")
plt.bar(x + 2 * w, carousel, w, label="Carousel",edgecolor = "black")

plt.xticks(x, age)
plt.ylabel("Conversion Rate (%)")
plt.title("Conversion Rates: Content Type & Age Group")
plt.legend(fontsize = 8, bbox_to_anchor=(0.72, 0.5, 0.5, 0.5))
plt.show()

## Conversion Rates: Content Type and Gender

In [ ]:
video = ad[ad['Content_Type'] == 'Video'].groupby('Target_Gender')['Conversion_Rate'].mean()
image = ad[ad['Content_Type'] == 'Image'].groupby('Target_Gender')['Conversion_Rate'].mean()
text = ad[ad['Content_Type'] == 'Text'].groupby('Target_Gender')['Conversion_Rate'].mean()
story = ad[ad['Content_Type'] == 'Story'].groupby('Target_Gender')['Conversion_Rate'].mean()
carousel = ad[ad['Content_Type'] == 'Carousel'].groupby('Target_Gender')['Conversion_Rate'].mean()

gender = np.sort(ad['Target_Gender'].unique())

x = np.arange(len(gender))
w = 0.15

plt.bar(x - 2 * w, video, w, label="Video",edgecolor = "black")
plt.bar(x - w, image, w, label="Image",edgecolor = "black")
plt.bar(x, text, w, label="Text",edgecolor = "black")
plt.bar(x + w, story, w, label="Story",edgecolor = "black")
plt.bar(x + 2 * w, carousel, w, label="Carousel",edgecolor = "black")

plt.xticks(x, gender)
plt.ylabel("Conversion Rate (%)")
plt.title("Conversion Rates: Content Type & Gender")
plt.legend(fontsize = 8, bbox_to_anchor=(0.72, 0.5, 0.5, 0.5))
plt.show()

## Conversion Rates: Content Type and Region

In [ ]:
video = ad[ad['Content_Type'] == 'Video'].groupby('Region')['Conversion_Rate'].mean()
image = ad[ad['Content_Type'] == 'Image'].groupby('Region')['Conversion_Rate'].mean()
text = ad[ad['Content_Type'] == 'Text'].groupby('Region')['Conversion_Rate'].mean()
story = ad[ad['Content_Type'] == 'Story'].groupby('Region')['Conversion_Rate'].mean()
carousel = ad[ad['Content_Type'] == 'Carousel'].groupby('Region')['Conversion_Rate'].mean()

region = np.sort(ad['Region'].unique())

x = np.arange(len(region))
w = 0.15

plt.bar(x - 2 * w, video, w, label="Video",edgecolor = "black")
plt.bar(x - w, image, w, label="Image",edgecolor = "black")
plt.bar(x, text, w, label="Text",edgecolor = "black")
plt.bar(x + w, story, w, label="Story",edgecolor = "black")
plt.bar(x + 2 * w, carousel, w, label="Carousel",edgecolor = "black")

plt.xticks(x, region)
plt.ylabel("Conversion Rate (%)")
plt.title("Conversion Rates: Content Type & Region")
plt.legend(fontsize = 8, bbox_to_anchor=(0.72, 0.5, 0.5, 0.5))
plt.show()

## Conversion Rates: Platform and Age Group

In [ ]:
Facebook = ad[ad['Platform'] == 'Facebook'].groupby('Target_Age')['Conversion_Rate'].mean()
Google = ad[ad['Platform'] == 'Google'].groupby('Target_Age')['Conversion_Rate'].mean()
Instagram = ad[ad['Platform'] == 'Instagram'].groupby('Target_Age')['Conversion_Rate'].mean()
LinkedIn = ad[ad['Platform'] == 'LinkedIn'].groupby('Target_Age')['Conversion_Rate'].mean()
Youtube = ad[ad['Platform'] == 'YouTube'].groupby('Target_Age')['Conversion_Rate'].mean()

age = np.sort(ad['Target_Age'].unique())

x = np.arange(len(age))
w = 0.15

plt.bar(x - 2 * w, Facebook, w, label="Facebook", color = "#1F4E9B",edgecolor = "black")
plt.bar(x - w, Google, w, label="Google", color = "#4DAB6E",edgecolor = "black")
plt.bar(x, Instagram, w, label="Instagram", color =  "#FB9AB5",edgecolor = "black")
plt.bar(x + w, LinkedIn, w, label="LinkedIn", color = "#3BA8E7",edgecolor = "black")
plt.bar(x + 2 * w, Youtube, w, label="Youtube", color = "#B2202C",edgecolor = "black")


plt.xticks(x, age)
plt.ylabel("Conversion Rate (%)")
plt.title("Conversion Rates: Platform & Age Group")
plt.legend(fontsize = 8, bbox_to_anchor=(0.72, 0.5, 0.5, 0.5))
plt.show()

## Conversion Rates: Platform and Gender

In [ ]:
Facebook = ad[ad['Platform'] == 'Facebook'].groupby('Target_Gender')['Conversion_Rate'].mean()
Google = ad[ad['Platform'] == 'Google'].groupby('Target_Gender')['Conversion_Rate'].mean()
Instagram = ad[ad['Platform'] == 'Instagram'].groupby('Target_Gender')['Conversion_Rate'].mean()
LinkedIn = ad[ad['Platform'] == 'LinkedIn'].groupby('Target_Gender')['Conversion_Rate'].mean()
Youtube = ad[ad['Platform'] == 'YouTube'].groupby('Target_Gender')['Conversion_Rate'].mean()

gender = np.sort(ad['Target_Gender'].unique())

x = np.arange(len(gender))
w = 0.15

plt.bar(x - 2 * w, Facebook, w, label="Facebook", color = "#1F4E9B",edgecolor = "black")
plt.bar(x - w, Google, w, label="Google", color = "#4DAB6E",edgecolor = "black")
plt.bar(x, Instagram, w, label="Instagram", color =  "#FB9AB5",edgecolor = "black")
plt.bar(x + w, LinkedIn, w, label="LinkedIn", color = "#3BA8E7",edgecolor = "black")
plt.bar(x + 2 * w, Youtube, w, label="Youtube", color = "#B2202C",edgecolor = "black")


plt.xticks(x, gender)
plt.ylabel("Conversion Rate (%)")
plt.title("Conversion Rates: Platform & Gender")
plt.legend(fontsize = 8, bbox_to_anchor=(0.72, 0.5, 0.5, 0.5))
plt.show()

## Conversion Rates: Platform and Region

In [ ]:
Facebook = ad[ad['Platform'] == 'Facebook'].groupby('Region')['Conversion_Rate'].mean()
Google = ad[ad['Platform'] == 'Google'].groupby('Region')['Conversion_Rate'].mean()
Instagram = ad[ad['Platform'] == 'Instagram'].groupby('Region')['Conversion_Rate'].mean()
LinkedIn = ad[ad['Platform'] == 'LinkedIn'].groupby('Region')['Conversion_Rate'].mean()
Youtube = ad[ad['Platform'] == 'YouTube'].groupby('Region')['Conversion_Rate'].mean()

region = np.sort(ad['Region'].unique())

x = np.arange(len(region))
w = 0.15

plt.bar(x - 2 * w, Facebook, w, label="Facebook", color = "#1F4E9B",edgecolor = "black")
plt.bar(x - w, Google, w, label="Google", color = "#4DAB6E",edgecolor = "black")
plt.bar(x, Instagram, w, label="Instagram", color =  "#FB9AB5",edgecolor = "black")
plt.bar(x + w, LinkedIn, w, label="LinkedIn", color = "#3BA8E7",edgecolor = "black")
plt.bar(x + 2 * w, Youtube, w, label="Youtube", color = "#B2202C",edgecolor = "black")


plt.xticks(x, region)
plt.ylabel("Conversion Rate (%)")
plt.title("Conversion Rates: Platform & Region")
plt.legend(fontsize = 8, bbox_to_anchor=(0.72, 0.5, 0.5, 0.5))
plt.show()

# Conclusions & Recommendations

### Data Collection
* There is a discrepancy in data collection/data generation: Conversion Rates and Click-Through Rates exceed 100%. Additionally, CTR does not follow the standard formula/parameters.

>  **Recommendation**: It would be ideal to check on data collection measures to ensure data quality. In addition, other variables may also be added such as unique identifier for each user to make sure click to conversion ratio accurately represents the sample.

### Overall Campaign Success
* The campaign has an overall positive success, with different variables and target demographics having success rate higher than 80%.
  
* There is not much notable differences among platform success. However, Google has the highest success rate (94.55%) with Instagram being the last (87.11%)


* Similarly, there is not much difference in the success rates of different content types, Carousel having the highest success rate (94%) and text with the lowest (87.83%). However, despite being lowest, 6 out of the top performing campaigns (in terms of conversion) are text-type contents.

> **Recommendation**: For text-type campaigns, it would be ideal for the marketing team to explore what components are in the 6 campaigns that are among the top 10 campaigns with highest conversion rates. Albeit having the "lowest" success rate among the content types, the said top 10 may be studied further to understand the potential it has that contributes to high conversion rates.


### Budget and Duration
* Both budget and duration have very little to almost no correlation with conversion rates.

> **Recommendation**: Increasing (nor decreasing) the budget and  duration doesn't really relate to higher/lower conversion rate so this shouldn't be the main focus of improving the campaigns. Instead the marketing team may explore the effective strategies that they are currently doing to continue their success.

### Content Type & Demographics
* Video: Works well for younger audience (18 - 24) but not so much for 55+ in terms of conversion rates. This also worked well for US, Germany, and Canada.
* Image: Relatively similar conversion rates among age group. This also seemed to be the best choice for UK
* Text: This content type seems to be have really good conversion rates among all ages, especially for 18-24 and 55+.
* Story: Relatively lower conversion rates for younger audience.
* Carousel: Relatively lower conversion rates for 35-54, but surprisingly worked well for 55+. This also has the highest conversion rates for India, having big difference compared to the other content types.
* Note: For gender, the difference among conversion rates are relativelty similar.
> **Recommendation**: Marketing team may utilize the content type that notably works well for a population. For instance, carousel has visibly big difference among other types for the India region - we can then boost text posts for this region. Overall, text seems to be doing really great as well, so we can promote this more especially in US, and for those ages 18-24 and 55+.

### Platform & Demographics
* Facebook: Relatively good conversion rate among all groups, gender, and region. Especially high average conversion rate for Canada.
* Google: Relatively lower conversion rates in India.
* Instagram: Relatively similar conversion rates among all demographics.
* LinkedIn: Very good conversion rates for ages 18-24.
* Youtube: Relatively lower conversion rates for ages 45-54, but seemed to work really well for those ages 55+
> **Recommendation**: There are only few notable differences among demographics-wise conversion rates across each platforms. Most platforms have similar conversion rates. Hence, we could focus on either (1) improving the conversion rates per platform by using effective content types or (2) reduce reliance on platforms with lower engagement among specific demographics, such as YouTube for the 45–54 age group or Google for users in India.